# Part D — Policy RAG Validation

Tasks 1-4: Document chunking, TF-IDF retrieval, grounded answering, and evaluation

In [13]:
import sys
import importlib
from pathlib import Path

# Dynamically find project root
current_dir = Path.cwd()
project_root = None

# Check if we're in the project folder already
if (current_dir / "part_d").exists() or (current_dir / "notebooks").exists():
    project_root = current_dir
else:
    # Search parent directories
    for parent in current_dir.parents:
        if (parent / "part_d").exists() and (parent / "notebooks").exists():
            project_root = parent
            break

if project_root is None:
    raise RuntimeError("Could not find project root. Expected to find 'part_d' and 'notebooks' directories.")

# Add project root to path
sys.path.insert(0, str(project_root))

# Import and reload modules
import mock_llm
importlib.reload(mock_llm)

import part_d.policy_rag as policy_rag
importlib.reload(policy_rag)

from mock_llm import chat
import pandas as pd

print("✓ Imports successful")
print(f"✓ Project root: {project_root}")
print(f"✓ Working directory: {Path.cwd()}")

✓ Imports successful
✓ Project root: /Users/tamil-4827/MASAI_IIT_PATNA_AI_PROJECTS/Instamart_Copilot
✓ Working directory: /Users/tamil-4827/MASAI_IIT_PATNA_AI_PROJECTS/Instamart_Copilot/notebooks


## Task 1: Document Chunking

Verify chunking produces 15 total chunks with proper overlap

In [14]:
# Task 1: Chunking with overlap=10, chunk_size=40
chunks = policy_rag.chunk_documents(overlap_words=10, chunk_size=40)

print(f"✓ Total chunks: {len(chunks)} (expected 15)")
assert len(chunks) == 15, f"Expected 15 chunks, got {len(chunks)}"

# Count chunks per source
source_counts = {}
for chunk in chunks:
    source = chunk['source']
    source_counts[source] = source_counts.get(source, 0) + 1

print("\nChunks per policy:")
for source in sorted(source_counts.keys()):
    print(f"  {source}: {source_counts[source]}")

# Verify overlap: consecutive chunks should share 10 words
print("\nVerifying overlap between consecutive chunks from same source:")
for i in range(len(chunks) - 1):
    if chunks[i]['source'] == chunks[i+1]['source']:
        words_i = chunks[i]['text'].split()
        words_next = chunks[i+1]['text'].split()
        
        # Last 10 words of chunk i should match first 10 words of chunk i+1
        overlap_size = len(words_i) - 30  # chunk_size=40, advance=30
        if overlap_size >= 10:
            print(f"  Chunk {i} → {i+1}: {overlap_size} word overlap")

✓ Total chunks: 15 (expected 15)

Chunks per policy:
  delivery_policy.txt: 3
  escalation_policy.txt: 3
  payment_policy.txt: 3
  privacy_policy.txt: 2
  refund_policy.txt: 4

Verifying overlap between consecutive chunks from same source:
  Chunk 0 → 1: 10 word overlap
  Chunk 1 → 2: 10 word overlap
  Chunk 3 → 4: 10 word overlap
  Chunk 4 → 5: 10 word overlap
  Chunk 6 → 7: 10 word overlap
  Chunk 7 → 8: 10 word overlap
  Chunk 9 → 10: 10 word overlap
  Chunk 11 → 12: 10 word overlap
  Chunk 12 → 13: 10 word overlap
  Chunk 13 → 14: 10 word overlap


## Task 2: TF-IDF Retrieval

Verify retrieval returns top 3 most similar chunks

In [15]:
# Task 2: Build retriever
vectorizer, chunks_list = policy_rag.build_retriever(chunks)
print(f"✓ Vectorizer fitted on {len(chunks_list)} chunks")

# Test query
test_query = "refund policy late delivery"
retrieved = policy_rag.retrieve(test_query, k=3, vectorizer=vectorizer, chunks=chunks_list)

print(f"\nRetrieved for query: '{test_query}'")
print(f"Top 3 chunks (all should be from refund_policy.txt):")
for i, chunk in enumerate(retrieved, 1):
    print(f"\n  {i}. [{chunk['source']}] (score: {chunk['score']:.4f})")
    print(f"     {chunk['text'][:100]}...")

# Verify all top 3 are from refund_policy.txt
sources = [chunk['source'] for chunk in retrieved]
assert all(s == 'refund_policy.txt' for s in sources), f"Expected all from refund_policy.txt, got {sources}"
print(f"\n✓ All top 3 chunks from refund_policy.txt")

✓ Vectorizer fitted on 15 chunks

Retrieved for query: 'refund policy late delivery'
Top 3 chunks (all should be from refund_policy.txt):

  1. [refund_policy.txt] (score: 0.2719)
     order is delivered in more than 45 minutes, the customer receives a full refund of the order amount....

  2. [refund_policy.txt] (score: 0.2514)
     Refund Policy. A refund can be requested only for orders whose status is completed. Orders with stat...

  3. [refund_policy.txt] (score: 0.2454)
     late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support syste...

✓ All top 3 chunks from refund_policy.txt


## Task 3: Grounded Question Answering

Verify answer retrieves context and contains expected facts

In [16]:
# Task 3: Test answering
test_question = "What delivery time makes a completed order eligible for a full refund?"
answer_text, sources = policy_rag.answer(test_question, vectorizer=vectorizer, chunks=chunks_list)

print(f"Question: {test_question}")
print(f"\nAnswer: {answer_text}")
print(f"\nSources used: {sources}")

# Verify answer contains expected fact
expected_fact = "more than 45 minutes"
assert expected_fact in answer_text, f"Expected answer to contain '{expected_fact}', got: {answer_text}"
print(f"\n✓ Answer contains expected fact: '{expected_fact}'")

# Verify source is refund_policy.txt
assert 'refund_policy.txt' in sources, f"Expected refund_policy.txt in sources, got {sources}"
print(f"✓ Source includes refund_policy.txt")

Question: What delivery time makes a completed order eligible for a full refund?

Answer: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.

Sources used: ['refund_policy.txt', 'privacy_policy.txt']

✓ Answer contains expected fact: 'more than 45 minutes'
✓ Source includes refund_policy.txt


## Task 4: Evaluation with Overlap=10

Evaluate on all 14 questions from policy_eval.csv

In [17]:
# Task 4a: Evaluate with overlap=10
print("Running evaluation with overlap=10, chunk_size=40...\n")
eval_results_with_overlap = policy_rag.evaluate_retrieval(
    overlap_words=10,
    chunk_size=40
)

print(f"Retrieval Accuracy: {eval_results_with_overlap['retrieval_accuracy']:.1%}")
print(f"Grounding Accuracy: {eval_results_with_overlap['grounding_accuracy']:.1%}")
print(f"\nAnswerable questions: {eval_results_with_overlap['answerable_count']}")
print(f"Unanswerable questions: {eval_results_with_overlap['unanswerable_count']}")

# Expected: 12/12 retrieval, 12/12 grounding on answerable
assert eval_results_with_overlap['retrieval_accuracy'] == 1.0, "Expected 100% retrieval accuracy with overlap=10"
assert eval_results_with_overlap['grounding_accuracy'] == 1.0, "Expected 100% grounding accuracy with overlap=10"
print("\n✓ All 12 answerable questions retrieved and grounded correctly")
print("✓ All 2 unanswerable questions handled with fallback")

Running evaluation with overlap=10, chunk_size=40...

Retrieval Accuracy: 100.0%
Grounding Accuracy: 100.0%

Answerable questions: 12
Unanswerable questions: 2

✓ All 12 answerable questions retrieved and grounded correctly
✓ All 2 unanswerable questions handled with fallback


In [18]:
# Show details for overlap=10
print("Detailed results (overlap=10):")
print("\nAnswerable questions:")
for detail in eval_results_with_overlap['details']:
    if detail['expected_doc'] != 'NONE':
        status = "✓" if (detail['retrieval_success'] and detail['grounding_success']) else "✗"
        print(f"{status} Q: {detail['question'][:60]}...")
        print(f"  Retrieval: {detail['retrieval_success']}, Grounding: {detail['grounding_success']}")
        if not detail['grounding_success']:
            print(f"  Expected: '{detail['expected_fact']}'")
            print(f"  Got: '{detail['answer'][:80]}...'")

print("\nUnanswerable questions:")
for detail in eval_results_with_overlap['details']:
    if detail['expected_doc'] == 'NONE':
        status = "✓" if detail['grounding_success'] else "✗"
        print(f"{status} Q: {detail['question'][:60]}...")
        print(f"  Answer: {detail['answer'][:80]}...")

Detailed results (overlap=10):

Answerable questions:
✓ Q: What delivery time makes a completed order eligible for a fu...
  Retrieval: True, Grounding: True
✓ Q: Who must approve a refund above INR 1000?...
  Retrieval: True, Grounding: True
✓ Q: How many days do I have to raise a refund request?...
  Retrieval: True, Grounding: True
✓ Q: What is the delivery fee on small orders?...
  Retrieval: True, Grounding: True
✓ Q: On orders of what value is delivery free?...
  Retrieval: True, Grounding: True
✓ Q: Why does the electronics delivery need a one time password?...
  Retrieval: True, Grounding: True
✓ Q: Is cash on delivery available for a large order?...
  Retrieval: True, Grounding: True
✓ Q: How soon is money returned when a payment fails but is deduc...
  Retrieval: True, Grounding: True
✓ Q: Can wallet balance be refunded as cash?...
  Retrieval: True, Grounding: True
✓ Q: When are delivery addresses deleted from active systems?...
  Retrieval: True, Grounding: True
✓ Q: How fa

## Task 4b: Evaluation with Overlap=0

Re-evaluate without overlap to demonstrate failure on boundary-split phrases

In [19]:
# Task 4b: Evaluate with overlap=0 (no overlap)
print("Running evaluation with overlap=0 (no overlap)...\n")
eval_results_no_overlap = policy_rag.evaluate_retrieval(
    overlap_words=0,
    chunk_size=40
)

print(f"Retrieval Accuracy: {eval_results_no_overlap['retrieval_accuracy']:.1%}")
print(f"Grounding Accuracy: {eval_results_no_overlap['grounding_accuracy']:.1%}")

# With overlap=0, we expect at least one failure
# because phrases may be split across chunk boundaries
print(f"\nComparison:")
print(f"  With overlap=10: {eval_results_with_overlap['grounding_accuracy']:.1%}")
print(f"  With overlap=0:  {eval_results_no_overlap['grounding_accuracy']:.1%}")

if eval_results_no_overlap['grounding_accuracy'] < eval_results_with_overlap['grounding_accuracy']:
    print("\n✓ Overlap=0 shows lower accuracy (demonstrates impact of chunk boundaries)")
else:
    print(f"\n⚠ No accuracy difference observed")

Running evaluation with overlap=0 (no overlap)...

Retrieval Accuracy: 100.0%
Grounding Accuracy: 85.7%

Comparison:
  With overlap=10: 100.0%
  With overlap=0:  85.7%

✓ Overlap=0 shows lower accuracy (demonstrates impact of chunk boundaries)


In [20]:
# Find and explain failures with overlap=0
print("Analyzing failures with overlap=0:\n")

failures = []
for i, detail in enumerate(eval_results_no_overlap['details']):
    detail_overlap10 = eval_results_with_overlap['details'][i]
    
    # Compare: did this question fail with overlap=0 but pass with overlap=10?
    if (detail_overlap10['grounding_success'] and not detail['grounding_success']):
        failures.append(detail)
        print(f"Question: {detail['question']}")
        print(f"Expected fact: {detail['expected_fact']}")
        print(f"Answer: {detail['answer'][:120]}")
        print(f"\nExplanation: Without overlap, the phrase '{detail['expected_fact']}' may be split")
        print(f"             across chunk boundaries, preventing the LLM from seeing both parts together.")
        print()

if not failures:
    print("No failures detected with overlap=0. Policies may be small enough that phrases don't split.")
else:
    print(f"\n✓ Identified {len(failures)} failure(s) demonstrating TF-IDF limitation with phrase splitting")

Analyzing failures with overlap=0:

Question: What delivery time makes a completed order eligible for a full refund?
Expected fact: more than 45 minutes
Answer: receives a full refund of the order amount.

Explanation: Without overlap, the phrase 'more than 45 minutes' may be split
             across chunk boundaries, preventing the LLM from seeing both parts together.

Question: How soon is money returned when a payment fails but is deducted?
Expected fact: 48 hours
Answer: If a payment fails but money is deducted, the amount is returned automatically

 
1000 must be approved by a human suppo

Explanation: Without overlap, the phrase '48 hours' may be split
             across chunk boundaries, preventing the LLM from seeing both parts together.


✓ Identified 2 failure(s) demonstrating TF-IDF limitation with phrase splitting


## Summary

All Task 1-4 acceptance criteria verified:

In [21]:
print("✓ Task 1: Chunking")
print(f"  - 15 total chunks created")
print(f"  - Distribution: {source_counts}")

print("\n✓ Task 2: Retrieval")
print(f"  - TF-IDF vectorizer fitted on {len(chunks_list)} chunks")
print(f"  - Query '{test_query}' returns all top 3 from refund_policy.txt")

print("\n✓ Task 3: Answering")
print(f"  - Answers grounded in retrieved context")
print(f"  - Expected facts included in answers")

print("\n✓ Task 4: Evaluation")
print(f"  - With overlap=10: {eval_results_with_overlap['retrieval_accuracy']:.1%} retrieval, {eval_results_with_overlap['grounding_accuracy']:.1%} grounding")
print(f"  - With overlap=0:  {eval_results_no_overlap['retrieval_accuracy']:.1%} retrieval, {eval_results_no_overlap['grounding_accuracy']:.1%} grounding")
print(f"  - 12/12 answerable questions handled correctly (with overlap)")
print(f"  - 2/2 unanswerable questions handled with fallback")

print("\n✓✓✓ PART D FILE 1 VALIDATION COMPLETE ✓✓✓")

✓ Task 1: Chunking
  - 15 total chunks created
  - Distribution: {'delivery_policy.txt': 3, 'escalation_policy.txt': 3, 'payment_policy.txt': 3, 'privacy_policy.txt': 2, 'refund_policy.txt': 4}

✓ Task 2: Retrieval
  - TF-IDF vectorizer fitted on 15 chunks
  - Query 'refund policy late delivery' returns all top 3 from refund_policy.txt

✓ Task 3: Answering
  - Answers grounded in retrieved context
  - Expected facts included in answers

✓ Task 4: Evaluation
  - With overlap=10: 100.0% retrieval, 100.0% grounding
  - With overlap=0:  100.0% retrieval, 85.7% grounding
  - 12/12 answerable questions handled correctly (with overlap)
  - 2/2 unanswerable questions handled with fallback

✓✓✓ PART D FILE 1 VALIDATION COMPLETE ✓✓✓
